## ⚠ Save a copy to your Drive first

**This notebook is fetched fresh from GitHub every time you open the link.** Any edits you make here — settings, code, hyperparameters — **will be LOST when you close the tab** unless you save a copy.

**To keep your edits:**

1. **File → Save a copy in Drive** (top menu)
2. Re-open the saved copy via **File → Open notebook → Recent** or your Google Drive next time

The saved copy is yours to edit; the GitHub link always opens fresh.

# Train a Tabular Regressor for IGNODE

**Maintained by:** IGNODE  
**Last verified:** May 2026 against Python 3.11, XGBoost 2.1, scikit-learn 1.5  
**Runtime:** under 2 minutes on Colab's free CPU tier (no GPU needed)

Train a tabular regression model — predict numeric values like price, remaining life, or temperature. Choose between a single algorithm, a custom sweep, or AutoML in the **Settings** cell.

## What you'll need

- A `.csv` file with a **header row**, one column per feature, and one **numeric label column** (the value you want to predict)
- A Google account to run this notebook in Colab

## What this notebook produces

1. `model.onnx` — the trained model in ONNX format
2. `feature_columns.json` — sidecar metadata
3. The exact strings you'll paste into the IGNODE Custom Model Upload wizard

## Steps

1. **Runtime → Run all** (top menu) — or run each cell in order with Shift+Enter
2. When the upload cell prompts, drop your CSV file
3. Edit the **Settings** cell if you want a different algorithm or hyperparameters
4. After training, download `model.onnx` and the metadata, then go to **IGNODE → ML Factory → Custom Models → + Upload ML Model**

## Quick start

### To try it right now (no setup needed)

1. **Runtime → Run all** at the top of Colab
2. Wait ~30 seconds for dependencies + ~60 seconds for AutoML to explore 4 tree-based algorithms
3. The last cell automatically downloads `model.onnx` + the sidecar JSON to your laptop — that's the AutoML winner on the sample IoT data

### To train on YOUR data

You only need to edit **two values**:

| Step | Cell | What to change |
|---|---|---|
| 1 | **Load data** cell | `SAMPLE_DATASET = 'equipment_rul_regression'` → `SAMPLE_DATASET = None` |
| 2 | **Settings** cell | `LABEL_COLUMN = 'RemainingLife'` → `LABEL_COLUMN = 'your_numeric_target_column'` |

The target column **must be numeric** for regression.

### Three ways to control which algorithm trains

This notebook accepts three shapes for `ALGORITHM` in the Settings cell:

| `ALGORITHM` value | Result |
|---|---|
| `"auto"` | AutoML over 4 tree-based algorithms (default) |
| `"lightgbm"` (or another name) | Train just that algorithm — fast |
| `["lightgbm", "xgboost"]` | Custom sweep — train each, pick the winner by R² |

Linear regression flavors (`"ridge"` and `"elastic_net"`) are available in single + sweep modes but not whitelisted for `"auto"` mode — FLAML doesn't expose them first-class for regression.

### What you get at the end

- `model.onnx` — the winning regression model
- `feature_columns.json` — input contract (`{feature_columns, label_columns}`)

No `class_labels.json` — regression predicts numbers, not classes.

Drop the artifacts into **IGNODE → ML Factory → Custom Models → + Upload ML Model**.

## 1. Setup — install pinned dependencies

Pinned versions so this notebook still works in 6 months.

In [ ]:
!pip install --quiet \
    lightgbm==4.5.0 \
    xgboost==2.1.2 \
    scikit-learn==1.5.2 \
    flaml==2.3.2 \
    onnx==1.21.0 \
    onnxmltools==1.16.0 \
    skl2onnx==1.20.0 \
    onnxconverter-common==1.14.0 \
    onnxruntime==1.23.2

import sys
import lightgbm as lgb
import xgboost as xgb
import sklearn
import flaml
import onnx
print(f'Python:       {sys.version.split()[0]}')
print(f'LightGBM:     {lgb.__version__}')
print(f'XGBoost:      {xgb.__version__}')
print(f'scikit-learn: {sklearn.__version__}')
print(f'FLAML:        {flaml.__version__}')
print(f'onnx:         {onnx.__version__}')

## 2. Load data

This cell ships set to a small **IoT sample dataset** so the notebook runs end-to-end out of the box — just hit **Runtime → Run all** and you'll have a trained model in a couple of minutes.

When you're ready to use your own data:

1. Set `SAMPLE_DATASET = None` in the cell below — the cell will then prompt you to upload a CSV
2. Update `LABEL_COLUMN` in the Settings cell further down to the numeric column you want to predict

The default sample is `equipment_rul_regression.csv` — ~90 rows of industrial equipment telemetry (CycleCount, Temperature, Pressure, RPM, Vibration, OilTemp, CoolantFlow, PowerOutput, AmbientTemp, LoadPercent) with a `RemainingLife` label (numeric, cycles).

In [ ]:
# ───────── EDIT THIS ─────────
# The notebook ships with a sample IoT dataset so it runs end-to-end.
# Set SAMPLE_DATASET = None to upload your own CSV instead.
SAMPLE_DATASET = 'equipment_rul_regression'
# Available samples (regression):
#   'equipment_rul_regression'   — equipment telemetry, label='RemainingLife'
#   'building_energy_regression' — building features, label='HeatingLoad'
# ────────────────────────────

import pandas as pd

if SAMPLE_DATASET:
    url = f'https://raw.githubusercontent.com/IGNODE-CONNECT/ignode-collab/main/examples/{SAMPLE_DATASET}.csv'
    df = pd.read_csv(url)
    print(f'Loaded sample {SAMPLE_DATASET!r}: {df.shape[0]} rows x {df.shape[1]} columns')
else:
    from google.colab import files
    uploaded = files.upload()
    csv_path = next(iter(uploaded.keys()))
    df = pd.read_csv(csv_path)
    print(f'Loaded {csv_path}: {df.shape[0]} rows x {df.shape[1]} columns')

## 3. Inspect the data

Quick look at the columns, value distribution, and dtypes. If anything looks wrong (wrong column names, label column with non-numeric values, missing data), fix the CSV and re-upload before training.

In [ ]:
print('First 5 rows:')
display(df.head())

print('\nColumn dtypes:')
print(df.dtypes)

print('\nStatistics for numeric columns:')
display(df.describe())

## 4. Settings — pick your algorithm(s)

**Edit this cell to match your dataset.** Most users only touch `LABEL_COLUMN` and (optionally) `ALGORITHM`.

### Algorithm list (aligned with IGNODE's in-platform AutoML)

These are the sklearn equivalents of the algorithms IGNODE's in-platform AutoML tries for regression. A model trained here should perform comparably to one trained in ML Factory's ML Jobs tab.

| Name | sklearn / library equivalent | Mirrors IGNODE in-platform AutoML algorithm |
|---|---|---|
| `lightgbm` | LightGBM gradient-boosted trees | `LightGBM` |
| `xgboost` | XGBoost gradient-boosted trees | `FastTree` |
| `random_forest` | RandomForest | `FastForest` |
| `extra_trees` | ExtraTrees | _(no direct equivalent — adds diversity)_ |
| `ridge` | Ridge regression (L2 linear) | `OnlineGradientDescent` |
| `elastic_net` | ElasticNet (L1+L2 linear) | `SDCA` |

### `ALGORITHM` accepts three shapes

| Shape | Value | What runs |
|---|---|---|
| **AutoML** | `"auto"` | FLAML hyperparameter search over **all six** algorithms above |
| **Single** | `"lightgbm"` (or any one name from the list) | Just that algorithm with sensible defaults — fast direct fit |
| **Custom sweep** | `["lightgbm", "xgboost"]` (any list of 2+ names) | Train each with defaults, compare on held-out test set, pick the winner |

### Hyperparameters

- `AUTOML_TIME_BUDGET_SECONDS` — how long AutoML explores (only used when `ALGORITHM="auto"`). 30–120 seconds is plenty for typical CSVs.

In [ ]:
# ───────── EDIT THESE ─────────
# Default = the sample dataset's label column ('RemainingLife' for equipment_rul_regression).
# When you switch SAMPLE_DATASET = None above and upload your own CSV, change this
# to whichever numeric column you want to predict.
LABEL_COLUMN = 'RemainingLife'

# 'auto' = AutoML over all six algorithms
# Single string = direct fit (e.g. 'lightgbm', 'xgboost', 'random_forest',
#                              'extra_trees', 'ridge', 'elastic_net')
# List = custom sweep (e.g. ['lightgbm', 'xgboost'])
ALGORITHM = 'auto'

TEST_SIZE = 0.2
RANDOM_SEED = 42
AUTOML_TIME_BUDGET_SECONDS = 60   # only used when ALGORITHM='auto'
# ──────────────────────────────

# Sanity-check the label column.
if LABEL_COLUMN not in df.columns:
    raise ValueError(
        f"Label column '{LABEL_COLUMN}' not found in CSV. "
        f"Available columns: {list(df.columns)}"
    )

# Regression requires a numeric label column.
if not pd.api.types.is_numeric_dtype(df[LABEL_COLUMN]):
    raise ValueError(
        f"Label column '{LABEL_COLUMN}' must be numeric for regression. "
        f"Got dtype {df[LABEL_COLUMN].dtype}. Use the classifier notebook instead, "
        f"or pick a different LABEL_COLUMN."
    )

# Normalize ALGORITHM into one of three modes for the training cell.
SUPPORTED_ALGOS = {
    'lightgbm', 'xgboost', 'random_forest', 'extra_trees',
    'ridge', 'elastic_net',
}

if ALGORITHM == 'auto':
    algo_mode = 'auto'
    algo_value = None
elif isinstance(ALGORITHM, str):
    if ALGORITHM not in SUPPORTED_ALGOS:
        raise ValueError(
            f"ALGORITHM={ALGORITHM!r} is not recognized. "
            f"Use 'auto', one of {sorted(SUPPORTED_ALGOS)}, or a list of 2+ of those."
        )
    algo_mode = 'single'
    algo_value = ALGORITHM
elif isinstance(ALGORITHM, (list, tuple)):
    if len(ALGORITHM) < 2:
        raise ValueError("List form of ALGORITHM needs at least 2 entries; use a string for a single algorithm.")
    bad = [a for a in ALGORITHM if a not in SUPPORTED_ALGOS]
    if bad:
        raise ValueError(f"Unrecognized algorithm(s) {bad}. Valid: {sorted(SUPPORTED_ALGOS)}")
    if len(set(ALGORITHM)) != len(ALGORITHM):
        raise ValueError(f"Duplicate algorithm in list: {ALGORITHM}")
    algo_mode = 'sweep'
    algo_value = list(ALGORITHM)
else:
    raise TypeError(
        f"ALGORITHM must be 'auto', a string, or a list — got {type(ALGORITHM).__name__}"
    )

print(f"Label column: '{LABEL_COLUMN}' (dtype: {df[LABEL_COLUMN].dtype})")
print(f"Mode:         {algo_mode}{' (' + str(algo_value) + ')' if algo_value else ''}")
print(f"\nLabel distribution (summary):")
print(df[LABEL_COLUMN].describe())

## 5. Prep the data

Auto-rename column names with spaces / punctuation (IGNODE's upload wizard rejects those) and split into train + test sets. Note: regression has no class labels to encode.

In [ ]:
import re
from sklearn.model_selection import train_test_split

def normalize(name):
    return re.sub(r'[^A-Za-z0-9_-]+', '_', name).strip('_')

rename_map = {c: normalize(c) for c in df.columns if c != normalize(c)}
if rename_map:
    print('Renamed columns (apply same fix in your source CSV for clarity):')
    for old, new in rename_map.items():
        print(f'  {old!r}  ->  {new!r}')
    df = df.rename(columns=rename_map)
    if LABEL_COLUMN in rename_map:
        LABEL_COLUMN = rename_map[LABEL_COLUMN]

X = df.drop(columns=[LABEL_COLUMN])
y = df[LABEL_COLUMN].astype(float).values
feature_columns = list(X.columns)

print(f'Features ({len(feature_columns)}): {feature_columns}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
)
print(f'Train: {X_train.shape[0]} rows. Test: {X_test.shape[0]} rows.')

## 6. Train

Branches on the `ALGORITHM` you picked in the Settings cell:

- **`auto`** runs FLAML over the whitelisted regression algorithms. Per-trial log + final winner.
- **Single** trains one model with sensible defaults.
- **Sweep** trains each requested algorithm with defaults, compares on test set, picks the winner.

In [ ]:
import time
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import r2_score

# Maps the user-facing algorithm name to:
#   - FLAML's internal name (for 'auto' mode)
#   - a factory that builds a sklearn-compatible regressor with sensible defaults
def _build(name):
    if name == 'lightgbm':
        return 'lgbm', lgb.LGBMRegressor(
            n_estimators=200, learning_rate=0.05, max_depth=-1,
            random_state=RANDOM_SEED, verbose=-1,
        )
    if name == 'xgboost':
        return 'xgboost', xgb.XGBRegressor(
            n_estimators=200, learning_rate=0.05, max_depth=6,
            random_state=RANDOM_SEED,
        )
    if name == 'random_forest':
        return 'rf', RandomForestRegressor(
            n_estimators=200, max_depth=None, random_state=RANDOM_SEED, n_jobs=-1,
        )
    if name == 'extra_trees':
        return 'extra_tree', ExtraTreesRegressor(
            n_estimators=200, max_depth=None, random_state=RANDOM_SEED, n_jobs=-1,
        )
    if name == 'ridge':
        return 'ridge', Ridge(alpha=1.0, random_state=RANDOM_SEED)
    if name == 'elastic_net':
        return 'elastic_net', ElasticNet(
            alpha=0.1, l1_ratio=0.5, random_state=RANDOM_SEED, max_iter=5000,
        )
    raise ValueError(f"Unknown algorithm: {name}")


t0 = time.time()
winner_name = None
winner_model = None

if algo_mode == 'auto':
    # AutoML over all six. No stacking — winner is always single + exportable.
    from flaml import AutoML
    automl = AutoML()
    automl.fit(
        X_train=X_train,
        y_train=y_train,
        task='regression',
        time_budget=AUTOML_TIME_BUDGET_SECONDS,
        # FLAML doesn't expose ridge / elastic_net by default; we only use it
        # for the tree-based whitelist here. Linear models fall under 'single'
        # or 'sweep' mode for customers who specifically want them.
        estimator_list=['lgbm', 'xgboost', 'rf', 'extra_tree'],
        metric='r2',
        seed=RANDOM_SEED,
        verbose=1,
    )
    winner_name = automl.best_estimator
    winner_model = automl.model.estimator
    print(f'\nAutoML winner: {winner_name}')
    print(f'AutoML best validation R²: {1 - automl.best_loss:.3f}')

elif algo_mode == 'single':
    winner_name, winner_model = _build(algo_value)
    winner_model.fit(X_train, y_train)
    print(f'Trained {algo_value} ({winner_name}).')

elif algo_mode == 'sweep':
    # Train each requested algorithm with defaults; pick the one with the highest
    # test-set R² as the winner.
    print('Sweep results:')
    print(f"  {'Algorithm':<25} {'test R²':<12} {'fit time (sec)'}")
    print('  ' + '─' * 60)
    best_score = -float('inf')
    for name in algo_value:
        flaml_name, model = _build(name)
        t_fit = time.time()
        model.fit(X_train, y_train)
        fit_sec = time.time() - t_fit
        score = r2_score(y_test, model.predict(X_test))
        marker = ' ←' if score > best_score else ''
        print(f"  {name:<25} {score:<12.3f} {fit_sec:.1f}{marker}")
        if score > best_score:
            best_score = score
            winner_name = flaml_name
            winner_model = model
    print(f'\nSweep winner: {winner_name} (test R² {best_score:.3f})')

print(f'\nTotal elapsed: {time.time() - t0:.1f} sec')

## 7. Evaluate

Regression metrics on the held-out test set: **RMSE** (root mean squared error — penalizes big misses), **MAE** (mean absolute error — easier to interpret), and **R²** (variance explained — 1.0 is perfect, 0 means no better than predicting the mean).

A scatter plot of predicted vs actual lets you spot systematic bias (predictions consistently too high or low for certain ranges).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = winner_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Test-set RMSE: {rmse:.3f}')
print(f'Test-set MAE:  {mae:.3f}')
print(f'Test-set R²:   {r2:.3f}')

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, alpha=0.6, s=20)
lim = (min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max()))
ax.plot(lim, lim, 'r--', linewidth=1, label='perfect prediction')
ax.set_xlabel(f'Actual {LABEL_COLUMN}')
ax.set_ylabel(f'Predicted {LABEL_COLUMN}')
ax.set_title(f'Predicted vs Actual (test set, R²={r2:.3f})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Export to ONNX

Each algorithm has its own ONNX converter. The cell below picks the right one based on the winner type so all three modes (auto / single / sweep) produce equivalent ONNX files for the same algorithm.

Opset 18 first (matches IGNODE's pinset). Falls back to 15 then 12 if a converter doesn't support 18 yet.

In [ ]:
from onnxconverter_common.data_types import FloatTensorType
import onnx
import os

initial_types = [('input', FloatTensorType([None, len(feature_columns)]))]

ARTIFACT_FILE = None   # set below — 'model.onnx', 'model.txt', or 'model.json'


def _export(converter, model_obj, opsets=(18, 15, 12)):
    """Try ONNX export at each opset; return the proto on first success,
    or None if all attempts fail (so the caller can try a native fallback)."""
    last_err = None
    for opset in opsets:
        try:
            m = converter(model_obj, initial_types=initial_types, target_opset=opset)
            print(f'  ONNX exported at opset {opset}')
            return m
        except Exception as ex:
            last_err = ex
            print(f'  ONNX opset {opset} failed ({type(ex).__name__}); trying lower')
    return None


# Dispatch by the FLAML-side winner name so the auto / single / sweep modes
# all land on the right ONNX converter.
onnx_model = None
if winner_name == 'lgbm':
    from onnxmltools.convert import convert_lightgbm
    onnx_model = _export(convert_lightgbm, winner_model)
elif winner_name == 'xgboost':
    from onnxmltools.convert import convert_xgboost
    onnx_model = _export(convert_xgboost, winner_model)
elif winner_name in ('rf', 'extra_tree', 'ridge', 'elastic_net'):
    from skl2onnx import convert_sklearn
    onnx_model = _export(convert_sklearn, winner_model)
else:
    raise RuntimeError(f'Unsupported winner type for ONNX export: {winner_name}')

if onnx_model is not None:
    onnx.save_model(onnx_model, 'model.onnx')
    ARTIFACT_FILE = 'model.onnx'
    print(f'\nSaved model.onnx ({len(onnx_model.SerializeToString()) / 1024:.1f} KB)')
else:
    # ─── Native fallback (LightGBM + XGBoost only) ───
    # IGNODE's Custom Model Upload wizard accepts these formats first-class
    # (lgbm_text + xgb_json loaders). Sklearn winners have no equivalent
    # fallback — if that happens, retry with ALGORITHM = "lightgbm" or
    # "xgboost" to get into a fallback-capable algorithm.
    if winner_name == 'lgbm':
        winner_model.booster_.save_model('model.txt')
        ARTIFACT_FILE = 'model.txt'
        print(f'\n⚠ All ONNX opsets failed; saved LightGBM native model.txt ({os.path.getsize("model.txt") / 1024:.1f} KB)')
        print('  When uploading, the wizard will detect the LightGBM text format automatically.')
    elif winner_name == 'xgboost':
        winner_model.save_model('model.json')
        ARTIFACT_FILE = 'model.json'
        print(f'\n⚠ All ONNX opsets failed; saved XGBoost native model.json ({os.path.getsize("model.json") / 1024:.1f} KB)')
        print('  When uploading, the wizard will detect the XGBoost JSON format automatically.')
    else:
        raise RuntimeError(
            f'ONNX export failed for winner {winner_name!r} and no native fallback is available '
            f'for sklearn-style algorithms. Retry with ALGORITHM = "lightgbm" or "xgboost" to get '
            f'a fallback-capable algorithm, or try the simple/ notebook variants.'
        )

## 9. Write sidecar file

Regression models don't need class labels (they predict numbers), so the only sidecar is `feature_columns.json` — the input contract IGNODE's inference runtime reads to know which columns are features and which is the target.

In [ ]:
import json

feature_sidecar = {
    'feature_columns': feature_columns,
    'label_columns': [LABEL_COLUMN],
}
with open('feature_columns.json', 'w') as f:
    json.dump(feature_sidecar, f, indent=2)

print('feature_columns.json:')
print(json.dumps(feature_sidecar, indent=2))

## 10. Download the model

In [ ]:
from google.colab import files
files.download(ARTIFACT_FILE)   # 'model.onnx' / 'model.txt' (LightGBM) / 'model.json' (XGBoost)
files.download('feature_columns.json')

## 11. Upload to IGNODE

Open your IGNODE portal and:

1. **Integrations → ML Factory**
2. Switch to the **Custom Models** tab
3. Click **+ Upload ML Model** (or open the **Add Model** dropdown → **Upload Custom Model**)
4. Drop your `model.onnx` file
5. In the metadata form:
    - **Task Type:** `Regression`
    - **Feature Columns:** paste from `feature_columns.json` → `feature_columns`
    - **Target Column:** paste from `feature_columns.json` → `label_columns` (single entry)
6. Review and click **Upload**

Then click **Open in Playground** to test it with sample inputs.

### Troubleshooting

| Error | Fix |
|---|---|
| `Label column must be numeric for regression` | Your `LABEL_COLUMN` has string values. Use the classifier notebook instead. |
| Upload rejected: "invalid column names" | Re-export your CSV with the renamed columns from Step 5 |
| Upload rejected: "duplicate name" | A model with that name already exists in your org — pick a different name or delete the existing one |
| Playground says "Field X required" | `LABEL_COLUMN` was set wrong. Re-train with the correct value |
| Predictions are systematically off | Look at the Predicted-vs-Actual scatter. Bias toward the mean = under-fitting (try `ALGORITHM="auto"` or a more powerful algorithm). Wild scatter = noise the model can't capture from the features alone. |

Need to retrain with different settings? Edit the **Settings** cell and **Runtime → Run all** again.

---

## Reusing this notebook for your own data

Two edits switch to your own dataset:

```python
# In the "Load data" cell:
SAMPLE_DATASET = None        # was 'equipment_rul_regression'

# In the "Settings" cell:
LABEL_COLUMN = 'YourColumn'  # was 'RemainingLife' — your NUMERIC target column
```

Everything else adapts automatically. The target column must be numeric — strings throw a clear error pointing at the classifier notebook.

### Picking the right ALGORITHM mode

| Your goal | Set `ALGORITHM` to |
|---|---|
| Best possible model, willing to wait ~60 sec | `"auto"` (AutoML over 4 tree-based algos) |
| Fast result with strong default | `"lightgbm"` |
| Compare 2-3 specific algorithms | `["lightgbm", "xgboost", "random_forest"]` |
| Linear baseline | `"ridge"` or `"elastic_net"` |
| Compare trees vs linear | `["lightgbm", "ridge"]` |

### Other optional tweaks (Settings cell)

| Want to change | Edit |
|---|---|
| AutoML time budget | `AUTOML_TIME_BUDGET_SECONDS = 120` (more time = better odds of finding stronger model) |
| Train/test ratio | `TEST_SIZE = 0.3` |
| Reproducibility | `RANDOM_SEED = <any int>` |

### Reading the Predicted-vs-Actual scatter

- **Tight cluster along the diagonal** = strong predictions
- **Banana shape** = systematic bias at certain value ranges (often a non-linear relationship the model can't capture — try `"lightgbm"` or `"xgboost"` if you were on a linear algorithm)
- **Wide cloud** = predictions are noisy; either need more data or different features

### Bringing this code into your own project

The code is vanilla Python except for two Colab helpers (`files.upload()`, `files.download()`). To run outside Colab:

1. Replace the `files.upload()` block with `df = pd.read_csv('/your/local/path.csv')`
2. Replace the `files.download(...)` calls with whatever your project does with output files
3. Everything else works as-is.

### Common errors

| Error | Fix |
|---|---|
| `Label column must be numeric for regression` | Use the classifier notebook instead — your target is string categories |
| `Label column 'X' not in CSV` | Check the column list printed by the inspect cell |
| `ALGORITHM=X is not recognized` | Use one of the names in the algorithm table above |
| Upload rejected: "invalid column names" | Re-export your CSV with the renamed columns shown in the prep cell |
| AutoML picked a weak algorithm | Increase `AUTOML_TIME_BUDGET_SECONDS` or try the sweep form `ALGORITHM = ['lightgbm', 'xgboost', 'random_forest']` |